In [30]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pydantic import BaseModel
import joblib
import os

train_df_encoded = pd.read_csv('../../data/processed/training/train_data_v0.csv')
val_df_encoded = pd.read_csv('../../data/processed/training/val_data_v0.csv')
test_df_encoded = pd.read_csv('../../data/processed/training/test_data_v0.csv')

print(f"Train set: {train_df_encoded.shape}")
print(f"Validation set: {val_df_encoded.shape}")
print(f"Test set: {test_df_encoded.shape}")

Train set: (6851, 22)
Validation set: (762, 22)
Test set: (846, 22)


In [31]:


# ==================== DATA PREPARATION ====================
# Use already loaded data from previous cells
X_train = train_df_encoded.drop('Price', axis=1).values
y_train = train_df_encoded['Price'].values
X_val = val_df_encoded.drop('Price', axis=1).values
y_val = val_df_encoded['Price'].values
X_test = test_df_encoded.drop('Price', axis=1).values
y_test = test_df_encoded['Price'].values

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

# ==================== STANDARDIZE FEATURES ====================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nScaler fitted and applied to all sets")

# ==================== TRAIN KNN MODEL ====================
# Grid search for best k
best_k = None
best_mse = float('inf')

k_values = [3, 5, 7, 9, 11, 15, 21]

print("\n--- Grid Search for best k ---")
for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance', metric='euclidean')
    knn.fit(X_train_scaled, y_train)
    
    y_val_pred = knn.predict(X_val_scaled)
    mse = mean_squared_error(y_val, y_val_pred)
    
    print(f"k={k}: MSE={mse:.2f}, RMSE={np.sqrt(mse):.2f}")
    
    if mse < best_mse:
        best_mse = mse
        best_k = k

print(f"\nBest k: {best_k} with validation MSE: {best_mse:.2f}")

# Train final KNN model with best k
knn_model = KNeighborsRegressor(n_neighbors=best_k, weights='distance', metric='euclidean')
knn_model.fit(X_train_scaled, y_train)

# ==================== EVALUATE MODEL ====================
print("\n--- MODEL EVALUATION ---")

# Training set evaluation
y_train_pred = knn_model.predict(X_train_scaled)
train_mse = mean_squared_error(y_train, y_train_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

print(f"\nTRAINING SET:")
print(f"  MSE:  {train_mse:.2f}")
print(f"  RMSE: {np.sqrt(train_mse):.2f}")
print(f"  MAE:  {train_mae:.2f}")
print(f"  R²:   {train_r2:.4f}")

# Validation set evaluation
y_val_pred = knn_model.predict(X_val_scaled)
val_mse = mean_squared_error(y_val, y_val_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

print(f"\nVALIDATION SET:")
print(f"  MSE:  {val_mse:.2f}")
print(f"  RMSE: {np.sqrt(val_mse):.2f}")
print(f"  MAE:  {val_mae:.2f}")
print(f"  R²:   {val_r2:.4f}")

# Test set evaluation
y_test_pred = knn_model.predict(X_test_scaled)
test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"\nTEST SET:")
print(f"  MSE:  {test_mse:.2f}")
print(f"  RMSE: {np.sqrt(test_mse):.2f}")
print(f"  MAE:  {test_mae:.2f}")
print(f"  R²:   {test_r2:.4f}")

# ==================== SAVE MODEL AND SCALER ====================
joblib.dump(knn_model, 'knn_model.pkl')
joblib.dump(scaler, 'knn_scaler.pkl')
print("\n✓ Model and scaler saved")

# Store feature names for later use
feature_names = list(train_df_encoded.drop('Price', axis=1).columns)
print(f"\nFeature names: {feature_names}")

Training set shape: (6851, 21)
Validation set shape: (762, 21)
Test set shape: (846, 21)

Scaler fitted and applied to all sets

--- Grid Search for best k ---
k=3: MSE=120429.80, RMSE=347.03
k=5: MSE=111382.65, RMSE=333.74
k=7: MSE=105751.10, RMSE=325.19
k=9: MSE=107106.05, RMSE=327.27
k=11: MSE=107055.61, RMSE=327.19
k=15: MSE=107558.07, RMSE=327.96
k=21: MSE=110161.35, RMSE=331.91

Best k: 7 with validation MSE: 105751.10

--- MODEL EVALUATION ---

TRAINING SET:
  MSE:  2085.58
  RMSE: 45.67
  MAE:  9.21
  R²:   0.9959

VALIDATION SET:
  MSE:  105751.10
  RMSE: 325.19
  MAE:  215.21
  R²:   0.7933

TEST SET:
  MSE:  100771.93
  RMSE: 317.45
  MAE:  206.32
  R²:   0.7965

✓ Model and scaler saved

Feature names: ['CPU Mark', 'GPU Mark', 'Monitor', 'Width', 'Height', 'RAM', 'Storage Amount', 'Weight', 'Brand_Acer', 'Brand_Apple', 'Brand_Asus', 'Brand_Dell', 'Brand_HP', 'Brand_LG', 'Brand_Lenovo', 'Brand_MSI', 'Brand_Microsoft', 'Brand_Other', 'OS_Other', 'OS_Windows 10', 'OS_Windows 1

In [32]:
# ==================== INFERENCE FUNCTION 1: FROM DATASET ====================
# This function takes already processed and encoded data (like from the dataset)

def infer_price_from_encoded_data(encoded_data: pd.DataFrame) -> float:
    """
    Predict laptop price from already encoded data (similar to training dataset format)
    
    Args:
        encoded_data: DataFrame with one row containing all encoded features
                     (same features as training data)
    
    Returns:
        Predicted price (float)
    """
    # Load model and scaler if not already in memory
    model = knn_model
    scaler_obj = scaler
    
    # Extract features and scale
    X = encoded_data.values
    X_scaled = scaler_obj.transform(X)
    
    # Predict
    price = model.predict(X_scaled)[0]
    
    return price


# Test Inference 1: Using a sample from test set
print("\n" + "="*60)
print("INFERENCE TEST 1: Using encoded data from test set")
print("="*60)
sample_encoded = test_df_encoded.drop('Price', axis=1).iloc[0:1]
actual_price = test_df_encoded['Price'].iloc[0]
predicted_price = infer_price_from_encoded_data(sample_encoded)
error = abs(predicted_price - actual_price)
error_pct = (error / actual_price) * 100

print(f"Sample features: {sample_encoded.values[0][:5]}...")  # Show first 5 features
print(f"Actual Price:    ${actual_price:.2f}")
print(f"Predicted Price: ${predicted_price:.2f}")
print(f"Error:           ${error:.2f} ({error_pct:.1f}%)")


print("\n" + "="*60)


INFERENCE TEST 1: Using encoded data from test set
Sample features: [15398.0 2152.0 13.6 2560 1664]...
Actual Price:    $1599.00
Predicted Price: $1632.33
Error:           $33.33 (2.1%)



In [33]:
# ==================== PYDANTIC MODEL FOR RAW INPUT ====================
from pydantic import BaseModel, Field
from typing import Optional, List

class LaptopRawInput(BaseModel):
    """Raw laptop specification input from user"""
    os: str = Field(..., description="Operating System (e.g., 'Windows 11', 'MacOS', 'Windows 10')")
    brand: str = Field(..., description="Brand (e.g., 'Dell', 'HP', 'Apple', 'Lenovo', 'ASUS')")
    cpu: str = Field(..., description="CPU name (e.g., 'AMD Ryzen 5 5050U', 'Intel Core i7 12700U')")
    gpu: str = Field(..., description="GPU name (e.g., 'AMD Radeon Graphics', 'NVIDIA RTX 3060')")
    monitor: float = Field(..., description="Screen size in inches (e.g., 15.6)")
    resolution: str = Field(..., description="Display resolution (e.g., '1920x1080', '2560x1600')")
    ram: str = Field(..., description="RAM (e.g., '8GB', '16GB', '1TB')")
    storage: str = Field(..., description="Storage (e.g., '256GB', '512GB', '1TB')")
    weight: float = Field(..., description="Weight in kg (e.g., 1.8, 2.5)")


# ==================== HELPER FUNCTION: MAP CPU/GPU TO MARKS ====================
# TO DO: Fill this function to get CPU Mark and GPU Mark scores from CPU/GPU names
def get_cpu_gpu_marks(cpu_name: str, gpu_name: str) -> tuple:
    """
    Map CPU and GPU names to their performance marks/scores
    
    Args:
        cpu_name: CPU name string (e.g., 'Intel Core i7 12700U')
        gpu_name: GPU name string (e.g., 'NVIDIA RTX 3060')
    
    Returns:
        tuple: (cpu_mark, gpu_mark) - performance scores
        
    Note: TO DO - Implement this function to use your CPU/GPU mark lookup tables
          You can use the fuzzy matching from map_cpu_gpu.py or your own method
          Example structure:
          - Load CPU mark lookup (from data/raw/cpu_gpu_mark/cpu_mark.csv)
          - Load GPU mark lookup (from data/raw/cpu_gpu_mark/gpu_mark.csv)
          - Use fuzzy matching to find closest match
          - Return the mark scores
    """
    # TODO: Implementation here
    # For now, returning placeholder values
    cpu_mark = 12675.0  # Placeholder
    gpu_mark = 2000.0    # Placeholder
    
    return cpu_mark, gpu_mark


# ==================== INFERENCE FUNCTION 2: FROM RAW HUMAN INPUT ====================

def infer_price_from_raw_input(laptop_spec: LaptopRawInput) -> dict:
    """
    Predict laptop price from raw human input
    
    Args:
        laptop_spec: LaptopRawInput object with raw specifications
    
    Returns:
        dict with predicted price and details
    """
    model = knn_model
    scaler_obj = scaler
    
    # Get CPU and GPU marks
    cpu_mark, gpu_mark = get_cpu_gpu_marks(laptop_spec.cpu, laptop_spec.gpu)
    
    # Parse resolution
    width, height = map(int, laptop_spec.resolution.split('x'))
    
    # Parse RAM and Storage (handle GB/TB)
    if laptop_spec.ram.upper().endswith('TB'):
        ram_gb = float(laptop_spec.ram[:-2]) * 1024
    else:
        ram_gb = float(laptop_spec.ram.replace('GB', '').strip())
    
    if laptop_spec.storage.upper().endswith('TB'):
        storage_gb = float(laptop_spec.storage[:-2]) * 1024
    else:
        storage_gb = float(laptop_spec.storage.replace('GB', '').strip())
    
    # Create featured data matching the training dataset format
    # Initialize with zeros for all features
    data_dict = {}
    
    # Set numeric features
    data_dict['CPU Mark'] = cpu_mark
    data_dict['GPU Mark'] = gpu_mark
    data_dict['Monitor'] = float(laptop_spec.monitor)
    data_dict['Width'] = width
    data_dict['Height'] = height
    data_dict['RAM'] = ram_gb
    data_dict['Storage Amount'] = storage_gb
    data_dict['Weight'] = float(laptop_spec.weight)
    
    # Initialize all categorical features to 0
    known_brands = ['Brand_Acer', 'Brand_Apple', 'Brand_Asus', 'Brand_Dell', 'Brand_HP', 
                    'Brand_LG', 'Brand_Lenovo', 'Brand_MSI', 'Brand_Microsoft', 'Brand_Other']
    known_os = ['OS_Other', 'OS_Windows 10', 'OS_Windows 11']
    
    for brand_col in known_brands:
        data_dict[brand_col] = 0
    for os_col in known_os:
        data_dict[os_col] = 0
    
    # Set brand (one-hot encoding)
    brand_clean = laptop_spec.brand.lower()
    brand_mapping = {
        'acer': 'Brand_Acer',
        'apple': 'Brand_Apple',
        'asus': 'Brand_Asus',
        'dell': 'Brand_Dell',
        'hp': 'Brand_HP',
        'lg': 'Brand_LG',
        'lenovo': 'Brand_Lenovo',
        'msi': 'Brand_MSI',
        'microsoft': 'Brand_Microsoft'
    }
    
    if brand_clean in brand_mapping:
        data_dict[brand_mapping[brand_clean]] = 1
    else:
        data_dict['Brand_Other'] = 1
    
    # Set OS (one-hot encoding)
    os_clean = laptop_spec.os.lower()
    if 'windows 11' in os_clean:
        data_dict['OS_Windows 11'] = 1
    elif 'windows 10' in os_clean:
        data_dict['OS_Windows 10'] = 1
    else:
        data_dict['OS_Other'] = 1
    
    # Create DataFrame with features in correct order
    X = pd.DataFrame([data_dict])[feature_names]
    
    # Scale and predict
    X_scaled = scaler_obj.transform(X.values)
    predicted_price = model.predict(X_scaled)[0]
    
    # Return results
    return {
        'predicted_price': float(predicted_price),
        'specification': laptop_spec.model_dump(),
        'processed_features': X.to_dict(),
        'cpu_mark': cpu_mark,
        'gpu_mark': gpu_mark
    }


# ==================== TEST INFERENCE FUNCTION 2 ====================
print("INFERENCE TEST 2: Using raw human input")
print("="*60)

# Example input (Windows 11, AMD Ryzen 5, AMD Radeon)
test_input = LaptopRawInput(
    os="Windows 11",
    brand="HP",
    cpu="AMD Ryzen 5 5050U",
    gpu="AMD Radeon Graphics",
    monitor=15.6,
    resolution="1920x1080",
    ram="16GB",
    storage="512GB",
    weight=1.8
)

result = infer_price_from_raw_input(test_input)
print(f"\nInput Specification:")
print(f"  OS:       {test_input.os}")
print(f"  Brand:    {test_input.brand}")
print(f"  CPU:      {test_input.cpu}")
print(f"  GPU:      {test_input.gpu}")
print(f"  Monitor:  {test_input.monitor}\"")
print(f"  Resolution: {test_input.resolution}")
print(f"  RAM:      {test_input.ram}")
print(f"  Storage:  {test_input.storage}")
print(f"  Weight:   {test_input.weight}kg")

print(f"\nProcessed Values:")
print(f"  CPU Mark: {result['cpu_mark']}")
print(f"  GPU Mark: {result['gpu_mark']}")

print(f"\n>>> PREDICTED PRICE: ${result['predicted_price']:.2f}")
print("="*60)

print("\n✓ Note: Replace 'get_cpu_gpu_marks()' implementation to get actual CPU/GPU mark scores")

INFERENCE TEST 2: Using raw human input

Input Specification:
  OS:       Windows 11
  Brand:    HP
  CPU:      AMD Ryzen 5 5050U
  GPU:      AMD Radeon Graphics
  Monitor:  15.6"
  Resolution: 1920x1080
  RAM:      16GB
  Storage:  512GB
  Weight:   1.8kg

Processed Values:
  CPU Mark: 12675.0
  GPU Mark: 2000.0

>>> PREDICTED PRICE: $726.64

✓ Note: Replace 'get_cpu_gpu_marks()' implementation to get actual CPU/GPU mark scores
